# Insurance Cost Prediction Using Linear Regression

**Objective:** To predict insurance charges using Linear Regression and create interfaces with Gradio and Flask.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
data = pd.read_csv("insurance.csv")

data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
print("Dataset shape:", data.shape)

Dataset shape: (1338, 7)


In [5]:
print("\nColumn names:")
print(data.columns)


Column names:
Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='str')


In [6]:
print("\nMissing values:")
print(data.isnull().sum())


Missing values:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64


### Data Preparation
We select the input features and the target column (`charges`) for prediction.

In [7]:
X = data.drop("charges", axis=1)
y = data["charges"]

In [9]:
print("Input features:")
print(X.head())

Input features:
   age     sex     bmi  children smoker     region
0   19  female  27.900         0    yes  southwest
1   18    male  33.770         1     no  southeast
2   28    male  33.000         3     no  southeast
3   33    male  22.705         0     no  northwest
4   32    male  28.880         0     no  northwest


In [10]:
print("\nTarget:")
print(y.head())


Target:
0    16884.92400
1     1725.55230
2     4449.46200
3    21984.47061
4     3866.85520
Name: charges, dtype: float64


In [11]:
X = pd.get_dummies(
    X,
    columns=["sex", "smoker", "region"],
    drop_first=True,
    dtype=int
)

X.head()

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,0,1,0,0,1
1,18,33.770,1,1,0,0,1,0
2,28,33.000,3,1,0,0,1,0
3,33,22.705,0,1,0,1,0,0
4,32,28.880,0,1,0,1,0,0


### Train-Test Split
We use 80% of the data for training and 20% for testing.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (1070, 8)
Testing data: (268, 8)


### Model Training and Evaluation
We train the Linear Regression model, make predictions, and evaluate its performance using MAE, MSE, RMSE, and R² score.

In [14]:
model = LinearRegression()

In [15]:
model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [16]:
y_pred = model.predict(X_test)

print("Actual charges:")
print(y_test.head().to_list())

print("\nPredicted charges:")
print(y_pred[:5])

Actual charges:
[9095.06825, 5272.1758, 29330.98315, 9301.89355, 33750.2918]

Predicted charges:
[ 8969.55027444  7068.74744287 36858.41091155  9454.67850053
 26973.17345656]


In [17]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error (MAE):", round(mae, 2))
print("Mean Squared Error (MSE):", round(mse, 2))
print("Root Mean Squared Error (RMSE):", round(rmse, 2))
print("R-squared (R2) Score:", round(r2, 4))

Mean Absolute Error (MAE): 4181.19
Mean Squared Error (MSE): 33596915.85
Root Mean Squared Error (RMSE): 5796.28
R-squared (R2) Score: 0.7836


In [18]:
def predict_charges(age, sex, bmi, children, smoker, region):

    input_data = pd.DataFrame([{
        "age": age,
        "bmi": bmi,
        "children": children,
        "sex_male": 1 if sex == "male" else 0,
        "smoker_yes": 1 if smoker == "yes" else 0,
        "region_northwest": 1 if region == "northwest" else 0,
        "region_southeast": 1 if region == "southeast" else 0,
        "region_southwest": 1 if region == "southwest" else 0
    }])

    input_data = input_data.reindex(columns=X.columns, fill_value=0)

    prediction = model.predict(input_data)[0]

    return round(float(prediction), 2)

## Gradio Interface
We create a simple web interface where users enter insurance details and get predicted charges.

In [19]:
import gradio as gr

In [20]:
demo = gr.Interface(
    fn=predict_charges,
    inputs=[
        gr.Number(label="Age", value=30),
        gr.Dropdown(["female", "male"], label="Sex"),
        gr.Number(label="BMI", value=25.0),
        gr.Number(label="Number of Children", value=0),
        gr.Dropdown(["no", "yes"], label="Smoker"),
        gr.Dropdown(
            ["northeast", "northwest", "southeast", "southwest"],
            label="Region"
        )
    ],
    outputs=gr.Number(label="Predicted Insurance Charges"),
    title="Insurance Cost Prediction",
    description="Predict insurance charges using Linear Regression."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Flask Interface
We create a webpage using Flask to enter insurance details and predict insurance charges.

In [21]:
from flask import Flask, request, render_template_string
from threading import Thread

In [22]:
app = Flask(__name__)

html_page = """
<!DOCTYPE html>
<html>
<head>
    <title>Insurance Cost Prediction</title>
</head>
<body>
    <h2>Insurance Cost Prediction</h2>

    <form method="POST">
        <label>Age:</label>
        <input type="number" name="age" value="30" required>
        <br><br>

        <label>Sex:</label>
        <select name="sex">
            <option value="female">Female</option>
            <option value="male">Male</option>
        </select>
        <br><br>

        <label>BMI:</label>
        <input type="number" step="any" name="bmi" value="25" required>
        <br><br>

        <label>Number of Children:</label>
        <input type="number" name="children" value="0" required>
        <br><br>

        <label>Smoker:</label>
        <select name="smoker">
            <option value="no">No</option>
            <option value="yes">Yes</option>
        </select>
        <br><br>

        <label>Region:</label>
        <select name="region">
            <option value="northeast">Northeast</option>
            <option value="northwest">Northwest</option>
            <option value="southeast">Southeast</option>
            <option value="southwest">Southwest</option>
        </select>
        <br><br>

        <button type="submit">Predict Charges</button>
    </form>

    {% if prediction is not none %}
        <h3>Predicted Insurance Charges: {{ prediction }}</h3>
    {% endif %}
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def home():

    prediction = None

    if request.method == "POST":

        age = int(request.form["age"])
        sex = request.form["sex"]
        bmi = float(request.form["bmi"])
        children = int(request.form["children"])
        smoker = request.form["smoker"]
        region = request.form["region"]

        prediction = predict_charges(
            age, sex, bmi, children, smoker, region
        )

    return render_template_string(
        html_page,
        prediction=prediction
    )

In [23]:
flask_thread = Thread(
    target=lambda: app.run(
        host="127.0.0.1",
        port=5000,
        debug=False,
        use_reloader=False
    ),
    daemon=True
)

flask_thread.start()

print("Flask application started.")
print("Open: http://127.0.0.1:5000")

Flask application started.
Open: http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Oct/2026 10:39:29] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Oct/2026 10:39:29] "GET /favicon.ico HTTP/1.1" 404 -
